# CMIB → Blender: demo bốn ô trên Google Colab

Notebook này chạy inference pretrained bằng repo [tree115/Motion-In-Betweening](https://github.com/tree115/Motion-In-Betweening.git), rồi dựng video Blender + FFmpeg. Pipeline: BVH LAFAN → bốn biến thể CMIB → motion.npz → comparison.mp4.

Notebook không huấn luyện lại model và không vá run_cmib.py. Cần checkpoint, đúng le_classes_.npy và BVH LAFAN test. Nhân vật là mannequin procedural; để giống mesh trong paper cần character mesh/rig có giấy phép, tương thích hoặc retarget sang skeleton LAFAN.

## 1. Mount Drive và cấu hình

Chỉnh đường dẫn, start frame và label nếu cần. Mặc định notebook tìm assets trong MyDrive/CMIB/assets.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, sys, json, shutil, subprocess, zipfile, io, importlib.util

drive.mount("/content/drive")
REPO_URL = "https://github.com/tree115/Motion-In-Betweening.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/Motion-In-Betweening")

DRIVE_ROOT = Path("/content/drive/MyDrive/CMIB")
ASSET_DIR = DRIVE_ROOT / "assets"
OUTPUT_DIR = DRIVE_ROOT / "runs" / "blender_demo"
ASSET_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT = ASSET_DIR / "train-3100.pt"
CLASSES = ASSET_DIR / "le_classes_.npy"
BVH = ASSET_DIR / "jumps1_subject5.bvh"
SOURCE_LABEL = "jumps"
SEMANTIC_LABEL = "dance"
START_FRAME = 0
ANCHOR = None                 # None = giữa đoạn; hoặc số frame, ví dụ 40
TRUST_LEGACY_CHECKPOINT = False

print("Repo:", REPO_URL, "| branch:", REPO_BRANCH)
print("Checkpoint:", CHECKPOINT)
print("Labels:", CLASSES)
print("BVH:", BVH)
print("Output:", OUTPUT_DIR)

## 2. Kiểm tra runtime

Inference chạy được trên CPU nhưng chậm hơn. Có thể bật GPU trong Colab qua Runtime → Change runtime type → T4 GPU. Blender dùng Cycles CPU để chạy headless ổn định.

In [ ]:
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Inference device:", DEVICE)

## 3. Clone repo của bạn

Code clone vào /content (vùng tạm); assets và kết quả ở Drive. URL được kiểm tra để tránh chạy nhầm repo.

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    origin = subprocess.run(
        ["git", "remote", "get-url", "origin"],
        cwd=REPO_DIR, check=True, capture_output=True, text=True,
    ).stdout.strip()
    if origin.rstrip("/") != REPO_URL.rstrip("/"):
        raise RuntimeError(f"Repo hiện có không đúng URL: {origin}")
    subprocess.run(["git", "pull", "--ff-only", "origin", REPO_BRANCH],
                   cwd=REPO_DIR, check=True)

required = [REPO_DIR / "demo.py", REPO_DIR / "tools" / "render_video.py"]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Thiếu script trong repo: {missing}")
print("Repo:", REPO_DIR)
print("Commit:", subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR,
    check=True, capture_output=True, text=True
).stdout.strip())

## 4. Cài dependency và Blender

Không cài lại PyTorch để giữ nguyên phiên bản CUDA Colab. Blender/FFmpeg chỉ cài nếu runtime chưa có.

In [ ]:
missing_python = [
    package for package in ("numpy", "scipy")
    if importlib.util.find_spec(package) is None
]
if missing_python:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_python],
                   check=True)

if shutil.which("blender") is None or shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "blender", "ffmpeg"],
                   check=True)

print(subprocess.run(["blender", "--version"], check=True,
                     capture_output=True, text=True).stdout.splitlines()[0])
print(subprocess.run(["ffmpeg", "-version"], check=True,
                     capture_output=True, text=True).stdout.splitlines()[0])

## 5. Nạp checkpoint, label và BVH

Đặt train-3100.pt, le_classes_.npy và jumps1_subject5.bvh trong thư mục assets ở cell 1. Nếu checkpoint/label thiếu, cell sau cho upload ZIP pretrained weights; ZIP chỉ đọc đúng các file cần thiết, không extract đường dẫn tùy ý.

Pretrained weights được README CMIB gốc liên kết tại https://works.do/FCqKVjy. Nếu chưa có BVH, tải một BVH LAFAN test hợp lệ rồi upload. LAFAN1 có điều khoản CC BY-NC-ND; kiểm tra license trước khi chia sẻ dữ liệu/video.

In [ ]:
if not CHECKPOINT.is_file() or not CLASSES.is_file():
    print("Upload ZIP có train-3100.pt và le_classes_.npy.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise RuntimeError("Hãy upload đúng một file ZIP pretrained weights.")
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as archive:
        members = [m for m in archive.infolist() if not m.is_dir()]
        checkpoints = [m for m in members if Path(m.filename).name == "train-3100.pt"]
        labels = [m for m in members if Path(m.filename).name == "le_classes_.npy"]
        if not checkpoints:
            all_pt = [m for m in members if m.filename.lower().endswith(".pt")]
            if len(all_pt) == 1:
                checkpoints = all_pt
            else:
                raise FileNotFoundError("ZIP cần train-3100.pt hoặc chỉ một file .pt.")
        if len(labels) != 1:
            raise FileNotFoundError("ZIP cần đúng một file le_classes_.npy.")
        for member, destination in ((checkpoints[0], CHECKPOINT), (labels[0], CLASSES)):
            with archive.open(member) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)
    print("Đã lưu weights vào Drive.")

if not BVH.is_file():
    uploaded_bvh = files.upload()
    bvh_names = [name for name in uploaded_bvh if name.lower().endswith(".bvh")]
    if len(bvh_names) != 1:
        raise RuntimeError("Hãy upload đúng một file BVH LAFAN.")
    BVH = ASSET_DIR / Path(bvh_names[0]).name
    BVH.write_bytes(uploaded_bvh[bvh_names[0]])

if not all(path.is_file() for path in (CHECKPOINT, CLASSES, BVH)):
    raise FileNotFoundError("Thiếu checkpoint/label/BVH; kiểm tra lại đường dẫn.")
print("Checkpoint:", CHECKPOINT, f"({CHECKPOINT.stat().st_size / 1024**2:.1f} MiB)")
print("Labels:", CLASSES)
print("BVH:", BVH)

## 6. Kiểm tra checkpoint và action labels

Label phải khớp embedding table của checkpoint. Nếu jumps/dance không có trong danh sách, sửa SOURCE_LABEL/SEMANTIC_LABEL ở cell 1.

In [ ]:
import numpy as np
import torch

ckpt_info = torch.load(
    CHECKPOINT, map_location="cpu", weights_only=not TRUST_LEGACY_CHECKPOINT
)
class_names = np.load(CLASSES, allow_pickle=False).astype(str).tolist()
print("Checkpoint keys:", sorted(ckpt_info.keys()))
for key in ("epoch", "horizon", "from_idx", "target_idx", "interpolation"):
    print(f"{key:16s}:", ckpt_info.get(key, "<missing>"))
print("Actions:", class_names)
if SOURCE_LABEL not in class_names or SEMANTIC_LABEL not in class_names:
    raise ValueError(
        f"Unknown action; source={SOURCE_LABEL!r}, semantic={SEMANTIC_LABEL!r}. "
        f"Available: {class_names}"
    )
del ckpt_info

## 7. Chạy inference CMIB

Sinh các panel Ground Truth, In-betweening, Pose-conditioned, Semantic-conditioned. Start frame tính từ 0; cửa sổ được chọn theo checkpoint.

In [ ]:
inference_dir = OUTPUT_DIR / "inference"
command = [
    sys.executable, str(REPO_DIR / "demo.py"),
    "--checkpoint", str(CHECKPOINT),
    "--classes", str(CLASSES),
    "--bvh", str(BVH),
    "--start", str(START_FRAME),
    "--source-label", SOURCE_LABEL,
    "--semantic-label", SEMANTIC_LABEL,
    "--device", DEVICE,
    "--output", str(inference_dir),
    "--export-bvh",
]
if ANCHOR is not None:
    command += ["--anchor", str(ANCHOR)]
if TRUST_LEGACY_CHECKPOINT:
    command.append("--trust-checkpoint")
print("Running:", " ".join(command))
subprocess.run(command, cwd=REPO_DIR, check=True)

MOTION_NPZ = inference_dir / "motion.npz"
if not MOTION_NPZ.is_file():
    raise FileNotFoundError(MOTION_NPZ)
print("Motion:", MOTION_NPZ)
print(json.loads((inference_dir / "metadata.json").read_text()))

## 8. Render preview MP4 trong Blender

Preview dùng tối đa 16 frame, 480×360 mỗi ô, Cycles CPU. Thư mục render được giữ trên Drive.

In [ ]:
PREVIEW_DIR = OUTPUT_DIR / "preview"
preview_command = [
    sys.executable, str(REPO_DIR / "tools" / "render_video.py"),
    "--motion", str(MOTION_NPZ),
    "--output", str(PREVIEW_DIR),
    "--width", "480", "--height", "360",
    "--samples", "8", "--threads", "4",
    "--engine", "CYCLES", "--max-frames", "16", "--overwrite",
]
subprocess.run(preview_command, cwd=REPO_DIR, check=True)
PREVIEW_MP4 = PREVIEW_DIR / "comparison.mp4"
print("Preview:", PREVIEW_MP4)

In [ ]:
from IPython.display import Video, display
display(Video(filename=str(PREVIEW_MP4), embed=True))

## 9. Render video đầy đủ

Chỉ chạy sau khi preview ổn. Bỏ giới hạn frame để render toàn bộ. Tăng samples để giảm noise (64 mặc định, 96 chất lượng cao hơn) nhưng thời gian render tăng.

In [ ]:
FINAL_DIR = OUTPUT_DIR / "final"
final_command = [
    sys.executable, str(REPO_DIR / "tools" / "render_video.py"),
    "--motion", str(MOTION_NPZ),
    "--output", str(FINAL_DIR),
    "--width", "960", "--height", "540",
    "--samples", "64", "--threads", "4",
    "--engine", "CYCLES", "--overwrite",
]
subprocess.run(final_command, cwd=REPO_DIR, check=True)
FINAL_MP4 = FINAL_DIR / "comparison.mp4"
print("Final video:", FINAL_MP4)

## 10. Ảnh pose chồng như hình tham khảo (tùy chọn)

Xuất ảnh tĩnh bốn ô, mỗi ô có bảy pose mẫu. MP4 ở phần trước mới là video chuyển động.

In [ ]:
GHOST_DIR = OUTPUT_DIR / "ghosts"
ghost_command = [
    sys.executable, str(REPO_DIR / "tools" / "render_video.py"),
    "--motion", str(MOTION_NPZ),
    "--output", str(GHOST_DIR),
    "--width", "960", "--height", "540",
    "--samples", "64", "--threads", "4",
    "--engine", "CYCLES", "--ghosts", "7", "--overwrite",
]
subprocess.run(ghost_command, cwd=REPO_DIR, check=True)
print("Pose montage:", GHOST_DIR / "comparison.png")

## Ghi chú / lỗi thường gặp

- Unknown action: sửa label theo danh sách Actions ở cell 6.
- Checkpoint/label không tương thích: dùng le_classes_.npy đi cùng chính checkpoint.
- BVH window mismatch: chọn BVH đủ dài hoặc đổi START_FRAME.
- Checkpoint pickle cũ: chỉ bật TRUST_LEGACY_CHECKPOINT khi tin nguồn.
- Muốn mesh giống paper: cần rigged human asset có giấy phép và bước retarget; repo hiện xuất mannequin procedural.
- Notebook này chạy inference pretrained; không tái huấn luyện 5000 epochs hay xác nhận metric paper.